# FinMark ML Project - Milestone 1: Data Preprocessing (Updated)
**Team: Group 5**

Updates from the original version:
- Added `enforce_valid_ranges()` to catch implausible-but-non-null values (Age < 0 or > 100; negative Amount) that previously survived cleaning as valid numbers.
- Range check now runs between type conversion and missing-value imputation, so flagged values get median-filled like any other missing value.
- `Amount` is now imputed with the **category-wise median** (by `ProductCategory`) instead of one global median, to avoid an artificial spike of identical values.
- Added a post-cleaning validation check that prints a warning if any Age/Amount values are still out of range after cleaning.

In [1]:
# ============================================================
# FinMark ML Project - Milestone 1: Data Preprocessing
# Team: Group 5
# Description: Load, inspect, clean, and save the three
# datasets for downstream EDA and modeling.
# ============================================================

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')  # Suppress pandas warnings

# ------------------------------------------------------------
# 1. Define file paths (exact csv filenames)
# ------------------------------------------------------------
FILE_PATHS = {
    'demographics': 'customer_demographics_contaminated.csv',
    'transactions': 'customer_transactions_contaminated.csv',
    'social': 'social_media_interactions_contaminated.csv'
}

# Team identifier for output files
TEAM_NAME = "group5"

# ------------------------------------------------------------
# 2. Helper functions for cleaning
# ------------------------------------------------------------

def safe_convert_to_numeric(series):
    """Convert a Series to numeric, coercing non-numeric values to NaN."""
    series = series.replace(['Free', 'free', 'None', 'Unknown', 'N/A', ''],
                            np.nan, regex=False)
    return pd.to_numeric(series, errors='coerce')

def clean_date_series(series):
    """Convert a Series of date strings to datetime, handling multiple formats within the same column."""
    formats = ['%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%d-%m-%Y', '%Y-%m-%d %H:%M:%S']

    series = series.astype(str).str.strip()
    result = pd.Series(pd.NaT, index=series.index, dtype='datetime64[ns]')
    unparsed = series.notna() & series.ne('nan') & result.isna()

    for fmt in formats:
        if not unparsed.any():
            break
        parsed = pd.to_datetime(series[unparsed], format=fmt, errors='coerce')
        result.loc[unparsed] = parsed
        unparsed = series.notna() & series.ne('nan') & result.isna()

    na_before = (series.isna() | series.eq('nan')).sum()
    na_after = result.isna().sum()
    if na_after > na_before:
        print(f"  Warning: {na_after - na_before} values didn't match any known format and were set to NaT.")

    return result

# ---- NEW: range-validation function (Fix 1) ----
def enforce_valid_ranges(df, name):
    """Flag implausible-but-non-null values as NaN so they get caught by imputation.

    Handles known sentinel/error values that survive type conversion because
    they are technically valid numbers (e.g. Age = -1 or 150, Amount = -100).
    """
    df = df.copy()
    if 'Age' in df.columns:
        invalid_age = (df['Age'] < 0) | (df['Age'] > 100)
        n = invalid_age.sum()
        if n > 0:
            print(f"  {name}: {n} implausible Age values (< 0 or > 100) set to NaN.")
            df.loc[invalid_age, 'Age'] = np.nan
    if 'Amount' in df.columns:
        invalid_amt = df['Amount'] < 0
        n = invalid_amt.sum()
        if n > 0:
            print(f"  {name}: {n} negative Amount values set to NaN.")
            df.loc[invalid_amt, 'Amount'] = np.nan
    return df

def inspect_dataset(df, name):
    """Print a detailed inspection of a dataset."""
    print(f"\n{'='*60}")
    print(f"INSPECTING: {name}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print("\nColumn data types:")
    print(df.dtypes.to_string())
    print("\nFirst 5 rows:")
    print(df.head())
    print("\nLast 5 rows:")
    print(df.tail())
    
    # Check if there are any numeric columns before calling describe
    numeric_cols = df.select_dtypes(include=['number']).columns
    if len(numeric_cols) > 0:
        print("\nSummary statistics (numeric columns):")
        print(df[numeric_cols].describe().to_string())
    else:
        print("\nNo numeric columns found – skipping summary statistics.")
    
    # Show unique values for string columns if few
    string_cols = df.select_dtypes(include=['string', 'object']).columns
    if len(string_cols) > 0:
        print("\nUnique values in string columns:")
        for col in string_cols:
            uniq = df[col].nunique()
            if uniq <= 10:
                print(f"  {col}: {uniq} unique -> {df[col].unique().tolist()}")
            else:
                print(f"  {col}: {uniq} unique (too many to list)")

def check_duplicates(df, name):
    dup = df.duplicated().sum()
    print(f"{name}: {dup} duplicate rows found.")
    return dup

def check_missing(df, name):
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({'Count': missing, '%': missing_pct})
    missing_df = missing_df[missing_df['Count'] > 0]
    if missing_df.empty:
        print(f"{name}: No missing values.")
    else:
        print(f"{name}: Missing values detected:")
        print(missing_df.to_string())
    return missing

def clean_dataset(df, name):
    """
    Apply preprocessing steps:
    1. Convert numeric columns (coerce errors).
    2. Convert date columns (coerce errors).
    3. Enforce plausible value ranges (flag sentinel/error values as NaN).
    4. Drop rows with >50% missing.
    5. Fill numeric missing with median (category-wise for Amount).
    6. Fill categorical missing with mode.
    7. Drop duplicates.
    8. Post-cleaning validation check.
    """
    print(f"\n{'='*60}")
    print(f"CLEANING: {name}")
    print(f"{'='*60}")

    # Make a copy to avoid altering original
    df_clean = df.copy()

    # ---- Step A: Detect and convert columns ----
    # Identify date-like columns
    date_keywords = ['date', 'datetime', 'timestamp', 'day', 'month', 'year']
    for col in df_clean.columns:
        col_lower = col.lower()
        if any(kw in col_lower for kw in date_keywords):
            if df_clean[col].dtype == 'object':
                df_clean[col] = clean_date_series(df_clean[col])
                print(f"  Converted '{col}' to datetime.")

    # Convert all object columns that are numeric-like to numeric
    for col in df_clean.select_dtypes(include=['object']).columns:
        converted = safe_convert_to_numeric(df_clean[col])
        if converted.notna().sum() > 0:
            df_clean[col] = converted
            print(f"  Converted '{col}' to numeric.")

    # ---- Step A.5: Enforce plausible value ranges (NEW - Fix 1 & 2) ----
    df_clean = enforce_valid_ranges(df_clean, name)

    # ---- Step B: Remove rows with >50% missing ----
    threshold = 0.5
    row_missing_pct = df_clean.isnull().sum(axis=1) / df_clean.shape[1]
    rows_to_drop = row_missing_pct[row_missing_pct > threshold].index
    if len(rows_to_drop) > 0:
        df_clean = df_clean.drop(index=rows_to_drop)
        print(f"  Dropped {len(rows_to_drop)} rows with >50% missing values.")

    # ---- Step C: Fill missing numeric columns with median (NEW - Fix 4: category-wise for Amount) ----
    numeric_cols = df_clean.select_dtypes(include=['number']).columns
    for col in numeric_cols:
        if df_clean[col].isnull().sum() > 0:
            if col == 'Amount' and 'ProductCategory' in df_clean.columns:
                df_clean[col] = df_clean.groupby('ProductCategory')[col].transform(
                    lambda s: s.fillna(s.median())
                )
                # catch any leftover NaN (e.g. category itself was missing)
                df_clean[col] = df_clean[col].fillna(df_clean[col].median())
                print(f"  Filled missing in '{col}' with category-wise (ProductCategory) median")
            else:
                median_val = df_clean[col].median()
                df_clean[col] = df_clean[col].fillna(median_val)
                print(f"  Filled missing in '{col}' with median = {median_val:.2f}")

    # ---- Step D: Fill missing string columns with mode ----
    string_cols = df_clean.select_dtypes(include=['object']).columns
    for col in string_cols:
        if df_clean[col].isnull().sum() > 0:
            mode_val = df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'Unknown'
            df_clean[col] = df_clean[col].fillna(mode_val)
            print(f"  Filled missing in '{col}' with mode = '{mode_val}'")

    # ---- Step E: Remove duplicate rows ----
    dup_count = df_clean.duplicated().sum()
    if dup_count > 0:
        df_clean = df_clean.drop_duplicates()
        print(f"  Removed {dup_count} duplicate rows.")

    # ---- Step F: Post-cleaning validation (NEW - Fix 3) ----
    if 'Age' in df_clean.columns and not df_clean['Age'].between(0, 100).all():
        print("  WARNING: Age values outside 0-100 still present after cleaning.")
    if 'Amount' in df_clean.columns and (df_clean['Amount'] < 0).any():
        print("  WARNING: Negative Amount values still present after cleaning.")

    # Final check
    print(f"  Final shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
    print(f"  Remaining missing values: {df_clean.isnull().sum().sum()}")
    return df_clean

def save_cleaned(df, base_name, team_name, suffix='_cleaned.csv'):
    """Save the cleaned DataFrame with a new filename including team name."""
    if df is not None:
        # Remove '_contaminated' and '.csv' from the base name
        clean_base = base_name.replace('_contaminated', '').replace('.csv', '')
        filename = f"{clean_base}_{team_name}{suffix}"
        df.to_csv(filename, index=False)
        print(f"  Saved: {filename}")
        return filename
    return None

# ------------------------------------------------------------
# 3. Main execution
# ------------------------------------------------------------

def main():
    print("="*60)
    print("FINMARK DATA PREPROCESSING START")
    print(f"TEAM: {TEAM_NAME.upper()}")
    print("="*60)

    # Load datasets
    dfs = {}
    for name, path in FILE_PATHS.items():
        try:
            df = pd.read_csv(path)
            dfs[name] = df
            print(f"✅ Loaded '{name}': {df.shape[0]} rows, {df.shape[1]} cols")
        except FileNotFoundError:
            print(f"❌ File not found: {path}")
            dfs[name] = None

    # If any dataset failed, exit
    if any(v is None for v in dfs.values()):
        print("\n⚠️  Some datasets could not be loaded. Please check file paths.")
        return

    # Inspect each dataset
    for name, df in dfs.items():
        inspect_dataset(df, name)

    # Check duplicates and missing (overall)
    print("\n" + "="*60)
    print("DUPLICATE & MISSING SUMMARY")
    print("="*60)
    for name, df in dfs.items():
        check_duplicates(df, name)
        check_missing(df, name)

    # Clean each dataset
    cleaned_dfs = {}
    for name, df in dfs.items():
        cleaned_dfs[name] = clean_dataset(df, name)

    # Save cleaned datasets
    print("\n" + "="*60)
    print("SAVING CLEANED DATASETS (GROUP 5)")
    print("="*60)
    saved_files = []
    for name, df_clean in cleaned_dfs.items():
        if df_clean is not None:
            base_filename = os.path.basename(FILE_PATHS[name])
            fname = save_cleaned(df_clean, base_filename, TEAM_NAME)
            saved_files.append(fname)

    print("\n" + "="*60)
    print("PREPROCESSING COMPLETE")
    print("="*60)
    print("Cleaned files saved:")
    for f in saved_files:
        print(f"  - {f}")
    print("\nNext steps: Proceed to EDA (Milestone 1) using the cleaned files.")

# ------------------------------------------------------------
# 4. Execute the script
# ------------------------------------------------------------
if __name__ == "__main__":
    main()


FINMARK DATA PREPROCESSING START
TEAM: GROUP5
✅ Loaded 'demographics': 3200 rows, 6 cols
✅ Loaded 'transactions': 3200 rows, 6 cols
✅ Loaded 'social': 3200 rows, 6 cols

INSPECTING: demographics
Shape: 3200 rows × 6 columns

Column data types:
CustomerID     str
Age            str
Gender         str
Location       str
IncomeLevel    str
SignupDate     str

First 5 rows:
                             CustomerID   Age  Gender           Location  \
0  9207fa75-5758-48d1-94ad-19c041e0520f  51.0  Female         Jensenberg   
1  5fb09cd8-a473-46f7-80bd-6e49cf509078   NaN  Female       Castilloport   
2  c139496e-cc89-498a-bd90-1fb4627b6cff  37.0    Male  Lake Jennifertown   
3  50118139-7264-428f-81cc-a25fddc5d6dd  44.0    Male          Port Carl   
4  7d1f2bbc-8d16-4fbc-9b37-ece3324e8ed4  50.0  Female          Jessebury   

  IncomeLevel  SignupDate  
0         Low  2022-11-17  
1        High  2020-07-21  
2         NaN  2021-01-01  
3      Medium  2024-06-10  
4        High  2023-08-24  

L